In [ ]:
import os
import math
import random
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from transformers.utils import logging as transformers_logging
transformers_logging.disable_progress_bar()
SEED = 42
IN_STEPS = 24
OUT_STEPS = 8
NUM_NODES = 79
TOP_M = 6
KNN = 3
GAMMA = 2.0
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = Path('./data')
TRAIN_PATH = DATA_DIR / 'train.npz'
VAL_PATH = DATA_DIR / 'val.npz'
TEST_PATH = DATA_DIR / 'test.npz'
ADJACENCY_PATH = DATA_DIR / 'adjacency.npy'
SCENIC_PROFILE_EMBEDDING_PATH = DATA_DIR / 'node_embeddings.npy'
PROFILE_ADJACENCY_PATH = DATA_DIR / 'semantic_adjacency.npy'
GPT2_PATH = Path('./pretrained/gpt2')
EXPERIMENT_DIR = Path('./outputs') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
BEST_MODEL_PATH = EXPERIMENT_DIR / 'best_model.pt'
BANK_PATH = EXPERIMENT_DIR / 'response_memory_bank_train_only.npz'
in_steps, out_steps, num_nodes = (IN_STEPS, OUT_STEPS, NUM_NODES)


In [ ]:
def load_dataset(path):
    with np.load(path, allow_pickle=False) as data:
        return {'x': np.asarray(data['x'], dtype=np.float32), 'y': np.asarray(data['y'], dtype=np.float32), 'event': np.asarray(data['event'], dtype=np.float32), 'future_time': np.array(data['future_time'])}
train_data = load_dataset(TRAIN_PATH)
val_data = load_dataset(VAL_PATH)
test_data = load_dataset(TEST_PATH)
x_tra, y_tra = (train_data['x'], train_data['y'])
x_val, y_val = (val_data['x'], val_data['y'])
x_tst, y_tst = (test_data['x'], test_data['y'])
time_tst = test_data['future_time']
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from transformers import GPT2Model
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def normalize_adj(adj):
    adj = adj.copy()
    adj = adj + np.eye(adj.shape[0])
    degree = np.sum(adj, axis=1)
    degree_mat = np.diag(np.power(degree, -0.5))
    return degree_mat @ adj @ degree_mat
spatial_adj = np.load(ADJACENCY_PATH).astype(np.float32)
norm_adj = normalize_adj(spatial_adj).astype(np.float32)
profile_adj = np.load(PROFILE_ADJACENCY_PATH).astype(np.float32)
np.fill_diagonal(profile_adj, 0.0)
scenic_profile_embedding = np.load(SCENIC_PROFILE_EMBEDDING_PATH).astype(np.float32)
profile_offdiag = profile_adj[~np.eye(profile_adj.shape[0], dtype=bool)]

class CausalTemporalProp(nn.Module):

    def __init__(self, st_dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.kernel_size = kernel_size
        self.norm = nn.LayerNorm(st_dim)
        self.depthwise = nn.Conv1d(st_dim, st_dim, kernel_size=kernel_size, groups=st_dim, padding=0)
        self.pointwise = nn.Conv1d(st_dim, st_dim, kernel_size=1)
        self.ffn = nn.Sequential(nn.Linear(st_dim, st_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(st_dim * 2, st_dim), nn.Dropout(dropout))

    def forward(self, h):
        residual = h
        h_norm = self.norm(h)
        B, T, N, D = h_norm.shape
        x = h_norm.permute(0, 2, 3, 1).reshape(B * N, D, T)
        x = F.pad(x, (self.kernel_size - 1, 0))
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = x.reshape(B, N, D, T).permute(0, 3, 1, 2)
        return residual + self.ffn(x)

class DynamicSpatialProp(nn.Module):

    def __init__(self, st_dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(st_dim)
        self.mix = nn.Sequential(nn.Linear(st_dim * 3, st_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(st_dim * 2, st_dim), nn.Dropout(dropout))

    def forward(self, h, adj):
        residual = h
        h_norm = self.norm(h)
        adj2 = torch.bmm(adj, adj)
        h_1 = torch.einsum('bij,btjd->btid', adj, h_norm)
        h_2 = torch.einsum('bij,btjd->btid', adj2, h_norm)
        h_mix = torch.cat([h_norm, h_1, h_2], dim=-1)
        return residual + self.mix(h_mix)

class NonCommutativeSTEncoder(nn.Module):

    def __init__(self, static_adj, profile_adj, scenic_profile_embedding, num_nodes, in_steps=24, st_dim=64, kernel_size=5, dropout=0.1, semantic_rank=16):
        super().__init__()
        self.num_nodes = num_nodes
        self.in_steps = in_steps
        self.semantic_rank = semantic_rank
        self.value_embedding = nn.Sequential(nn.Linear(1, st_dim), nn.LayerNorm(st_dim))
        self.register_buffer('static_adj', torch.tensor(static_adj, dtype=torch.float32))
        self.register_buffer('profile_adj', torch.tensor(profile_adj, dtype=torch.float32))
        offdiag_mask = torch.ones(num_nodes, num_nodes, dtype=torch.float32) - torch.eye(num_nodes, dtype=torch.float32)
        self.register_buffer('offdiag_mask', offdiag_mask)
        self.register_buffer('scenic_profile_embedding', torch.tensor(scenic_profile_embedding, dtype=torch.float32))
        self.temporal_prop = CausalTemporalProp(st_dim, kernel_size=kernel_size, dropout=dropout)
        self.spatial_prop = DynamicSpatialProp(st_dim, dropout=dropout)
        self.out_norm = nn.LayerNorm(st_dim)
        event_bottleneck_dim = 8
        event_hidden_dim = 32
        self.event_encoder = nn.Sequential(nn.LayerNorm(4096, elementwise_affine=False), nn.Linear(4096, event_bottleneck_dim, bias=False), nn.GELU(), nn.Linear(event_bottleneck_dim, event_hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.scenic_profile_norm = nn.LayerNorm(4096, elementwise_affine=False)
        self.scenic_profile_proj = nn.Linear(4096, semantic_rank, bias=False)
        self.event_time_head = nn.Linear(event_hidden_dim, in_steps)
        self.event_time_query = nn.Linear(event_hidden_dim, semantic_rank, bias=False)
        self.event_edge_src_query = nn.Linear(event_hidden_dim, semantic_rank, bias=False)
        self.event_edge_dst_query = nn.Linear(event_hidden_dim, semantic_rank, bias=False)
        self.event_branch_head = nn.Linear(event_hidden_dim, 2)
        nn.init.zeros_(self.event_time_head.weight)
        nn.init.zeros_(self.event_time_head.bias)
        nn.init.zeros_(self.event_time_query.weight)
        nn.init.zeros_(self.event_edge_src_query.weight)
        nn.init.zeros_(self.event_edge_dst_query.weight)
        nn.init.zeros_(self.event_branch_head.weight)
        nn.init.zeros_(self.event_branch_head.bias)

    def build_effective_adj(self, flow_input):
        return self.static_adj.unsqueeze(0).expand(flow_input.size(0), -1, -1)

    def encode_event(self, event_e):
        return self.event_encoder(event_e)

    def event_modulations(self, event_feat):
        scenic_key = self.scenic_profile_proj(self.scenic_profile_norm(self.scenic_profile_embedding))
        scenic_key = F.normalize(scenic_key, dim=-1)
        time_base = self.event_time_head(event_feat).unsqueeze(1)
        event_query = self.event_time_query(event_feat) / math.sqrt(self.semantic_rank)
        node_score = torch.matmul(event_query, scenic_key.t()).unsqueeze(-1)
        node_time_delta = torch.tanh(time_base + node_score)
        src_query = self.event_edge_src_query(event_feat) / math.sqrt(self.semantic_rank)
        dst_query = self.event_edge_dst_query(event_feat) / math.sqrt(self.semantic_rank)
        src_score = torch.matmul(src_query, scenic_key.t())
        dst_score = torch.matmul(dst_query, scenic_key.t())
        edge_raw = src_score.unsqueeze(2) + dst_score.unsqueeze(1)
        profile_prior = self.profile_adj * self.offdiag_mask
        edge_delta = profile_prior.unsqueeze(0) * torch.tanh(edge_raw)
        branch_logits = torch.tanh(self.event_branch_head(event_feat))
        branch_gate = torch.softmax(branch_logits, dim=-1)
        return (branch_logits, branch_gate, node_time_delta, edge_delta)

    def event_diagnostics(self, event_e):
        event_feat = self.encode_event(event_e)
        branch_logits, branch_gate, node_time_delta, edge_delta = self.event_modulations(event_feat)
        node_temporal_factor = 1.0 + node_time_delta
        return {'branch_logits': branch_logits, 'branch_gate': branch_gate, 'branch_share': branch_gate, 'time_delta': node_time_delta.mean(dim=1), 'node_time_delta': node_time_delta, 'edge_delta': edge_delta, 'temporal_factor': node_temporal_factor.mean(dim=1), 'node_temporal_factor': node_temporal_factor, 'edge_factor': 1.0 + edge_delta}

    def forward(self, flow_input, event_e):
        adj = self.build_effective_adj(flow_input)
        h0 = flow_input.transpose(1, 2).unsqueeze(-1)
        h0 = self.value_embedding(h0)
        event_feat = self.encode_event(event_e)
        _, branch_gate, node_time_delta, edge_delta = self.event_modulations(event_feat)
        h_t = self.temporal_prop(h0)
        temporal_factor = (1.0 + node_time_delta).transpose(1, 2).unsqueeze(-1)
        h_t_event = temporal_factor * h_t
        h_ts = self.spatial_prop(h_t_event, adj)
        edge_factor = 1.0 + edge_delta
        adj_st_event = torch.clamp(adj * edge_factor, min=0.0)
        adj_st_event = adj_st_event / (adj_st_event.sum(dim=-1, keepdim=True) + 1e-06)
        h_sp = self.spatial_prop(h0, adj_st_event)
        h_st = self.temporal_prop(h_sp)
        gate_ts = branch_gate[:, 0].view(-1, 1, 1, 1)
        gate_st = branch_gate[:, 1].view(-1, 1, 1, 1)
        h = gate_ts * h_ts + gate_st * h_st
        return self.out_norm(h)

class GPT2GraphBaseline(nn.Module):

    def __init__(self, static_adj, profile_adj=None, scenic_profile_embedding=None, num_nodes=79, in_steps=24, out_steps=8, st_dim=64, gpt2_path=GPT2_PATH, freeze_gpt2=True):
        super().__init__()
        self.num_nodes = num_nodes
        self.in_steps = in_steps
        self.out_steps = out_steps
        self.st_encoder = NonCommutativeSTEncoder(static_adj, profile_adj=profile_adj, scenic_profile_embedding=scenic_profile_embedding, num_nodes=num_nodes, in_steps=in_steps, st_dim=st_dim, kernel_size=5, dropout=0.1)
        self.scalar_out = nn.Sequential(nn.LayerNorm(st_dim), nn.Linear(st_dim, 1))
        self.delta_scale = nn.Parameter(torch.tensor(1.0))
        self.gpt2 = GPT2Model.from_pretrained(gpt2_path, local_files_only=True)
        if freeze_gpt2:
            for param in self.gpt2.parameters():
                param.requires_grad = False
        d_model = self.gpt2.config.n_embd
        self.temporal_tokenizer = nn.Sequential(nn.Linear(num_nodes, d_model), nn.LayerNorm(d_model))
        self.spatial_tokenizer = nn.Sequential(nn.Linear(in_steps, d_model), nn.LayerNorm(d_model))
        self.output_heads = nn.ModuleList([nn.Linear(d_model, num_nodes) for _ in range(out_steps)])

    def forward(self, flow_input, event_e):
        h_st = self.st_encoder(flow_input, event_e)
        raw_x = flow_input.transpose(1, 2)
        delta_x = self.scalar_out(h_st).squeeze(-1)
        x = raw_x + self.delta_scale * delta_x
        temporal_tokens = self.temporal_tokenizer(x)
        spatial_tokens = self.spatial_tokenizer(x.transpose(1, 2))
        gpt_tokens = torch.cat([temporal_tokens, spatial_tokens], dim=1)
        h = self.gpt2(inputs_embeds=gpt_tokens).last_hidden_state
        z = h.mean(dim=1)
        outs = [head(z) for head in self.output_heads]
        y = torch.stack(outs, dim=2)
        return y
model = GPT2GraphBaseline(norm_adj, profile_adj=profile_adj, scenic_profile_embedding=scenic_profile_embedding, num_nodes=num_nodes, in_steps=in_steps, out_steps=out_steps).to(DEVICE)
trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
total_params = sum((p.numel() for p in model.parameters()))
gpt2_trainable = [name for name, param in model.gpt2.named_parameters() if param.requires_grad]


In [ ]:
class EventConditionDataset(Dataset):

    def __init__(self, x, event, y=None):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.event = torch.as_tensor(event, dtype=torch.float32)
        self.y = None if y is None else torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.x.size(0)

    def __getitem__(self, idx):
        if self.y is None:
            return (self.x[idx], self.event[idx])
        return (self.x[idx], self.event[idx], self.y[idx])

def make_loader(data, batch_size=24, shuffle=False, with_targets=True):
    dataset = EventConditionDataset(data['x'], data['event'], data['y'] if with_targets else None)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
train_loader = make_loader(train_data, shuffle=True)
val_loader = make_loader(val_data)
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
import time

def _format_progress_bar(current, total, width=30):
    filled = int(width * current / total)
    return '[' + '=' * filled + '.' * (width - filled) + ']'

def run_epoch(model, loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_mae = 0.0
    total_count = 0
    for xb, eb, yb in loader:
        xb = xb.to(DEVICE)
        eb = eb.to(DEVICE)
        yb = yb.to(DEVICE)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            pred = model(xb, eb)
            loss = criterion(pred, yb)
            if train:
                loss.backward()
                optimizer.step()
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_mae += torch.mean(torch.abs(pred.detach() - yb)).item() * batch_size
        total_count += batch_size
    return (total_loss / total_count, total_mae / total_count)

def run_train_epoch_with_progress(model, loader, epoch, max_epochs):
    model.train(True)
    start = time.time()
    total_batches = len(loader)
    total_loss = 0.0
    total_mae = 0.0
    total_count = 0
    for batch_idx, (xb, eb, yb) in enumerate(loader, start=1):
        xb = xb.to(DEVICE)
        eb = eb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb, eb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_mae += torch.mean(torch.abs(pred.detach() - yb)).item() * batch_size
        total_count += batch_size
        elapsed = max(time.time() - start, 1e-08)
        ms_per_step = elapsed / batch_idx * 1000
        avg_loss = total_loss / total_count
        avg_mae = total_mae / total_count
        bar = _format_progress_bar(batch_idx, total_batches)
    return (total_loss / total_count, total_mae / total_count, time.time() - start)
best_val = float('inf')
best_state = None
patience = 50
wait = 0
max_epochs = 2500
for epoch in range(1, max_epochs + 1):
    train_loss, train_mae, elapsed = run_train_epoch_with_progress(model, train_loader, epoch, max_epochs)
    val_loss, val_mae = run_epoch(model, val_loader, train=False)
    total_batches = len(train_loader)
    ms_per_step = elapsed / max(total_batches, 1) * 1000
    bar = _format_progress_bar(total_batches, total_batches)
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
    if wait >= patience:
        break
if best_state is not None:
    model.load_state_dict(best_state)
    save_path = BEST_MODEL_PATH
    torch.save({'model_state_dict': model.state_dict(), 'num_nodes': num_nodes, 'in_steps': in_steps, 'out_steps': out_steps, 'best_val': best_val, 'model_name': 'GPT2GraphBaseline_EventConditionedSharedAdapterEventEdge'}, save_path)


In [ ]:
import itertools
import json
import math
import numpy as np
import pandas as pd
import torch

def make_time_features(values):
    dt = pd.to_datetime(pd.Series(np.asarray(values).reshape(-1)), errors='raise')
    minute = (dt.dt.hour * 60 + dt.dt.minute).to_numpy(np.float32)
    slot = (minute - 9 * 60) / 15.0
    weekday = dt.dt.weekday.to_numpy(np.float32)
    day = dt.dt.dayofyear.to_numpy(np.float32)
    month = dt.dt.month.to_numpy(np.float32)
    return np.stack([np.sin(2 * np.pi * slot / 36.0), np.cos(2 * np.pi * slot / 36.0), np.sin(2 * np.pi * weekday / 7.0), np.cos(2 * np.pi * weekday / 7.0), np.sin(2 * np.pi * day / 366.0), np.cos(2 * np.pi * day / 366.0), (month - 1.0) / 11.0, slot / 35.0], axis=1).astype(np.float32)

def retrieval_features(x, future_time):
    last = x[:, :, -1]
    mean = x.mean(axis=-1)
    std = x.std(axis=-1)
    trend = last - x[:, :, -6:-1].mean(axis=-1)
    early_late = x[:, :, -6:].mean(axis=-1) - x[:, :, :6].mean(axis=-1)
    mu = x.mean(axis=(1, 2), keepdims=True)
    sd = np.maximum(x.std(axis=(1, 2), keepdims=True), 1e-06)
    z = (x - mu) / sd
    last_z = z[:, :, -1]
    mean_z = z.mean(axis=-1)
    trend_z = last_z - z[:, :, -6:-1].mean(axis=-1)
    return np.concatenate([last, mean, std, trend, early_late, last_z, mean_z, trend_z, future_time.reshape(len(x), -1)], axis=1).astype(np.float32)
HISTORICAL_RETRIEVAL_DIM = 8 * NUM_NODES
SPECTRAL_RETRIEVAL_DIM = 0
FUTURE_TIME_RETRIEVAL_DIM = OUT_STEPS * 8
RETRIEVAL_KEEP_INDEX = np.r_[np.arange(HISTORICAL_RETRIEVAL_DIM), np.arange(HISTORICAL_RETRIEVAL_DIM + SPECTRAL_RETRIEVAL_DIM, HISTORICAL_RETRIEVAL_DIM + SPECTRAL_RETRIEVAL_DIM + FUTURE_TIME_RETRIEVAL_DIM)]


In [ ]:
def retrieve(bank, x, future_time, batch=128):
    memory_all = bank['memory_retrieval_feat'].astype(np.float32)
    mean_all = bank['retrieval_feat_mean'].astype(np.float32)
    std_all = np.maximum(bank['retrieval_feat_std'].astype(np.float32), 1e-06)
    memory = memory_all[:, RETRIEVAL_KEEP_INDEX]
    mean = mean_all[:, RETRIEVAL_KEEP_INDEX]
    std = std_all[:, RETRIEVAL_KEEP_INDEX]
    query = (retrieval_features(x, future_time) - mean) / std
    memory_norm = np.mean(memory * memory, axis=1)[None]
    indexes = []
    for start in range(0, len(x), batch):
        stop = min(start + batch, len(x))
        q = query[start:stop]
        distance = np.mean(q * q, axis=1, keepdims=True) + memory_norm - 2 * q @ memory.T / memory.shape[1]
        part = np.argpartition(distance, TOP_M - 1, axis=1)[:, :TOP_M]
        order = np.argsort(np.take_along_axis(distance, part, axis=1), axis=1)
        indexes.append(np.take_along_axis(part, order, axis=1))
    return bank['memory_residual'][np.concatenate(indexes)].astype(np.float32)


In [ ]:
def project_edges_and_triangles(base_residual, candidates):
    count, m, n, h = candidates.shape
    dim = n * h
    edge_output = np.empty((count, n, h), dtype=np.float32)
    complex_output = np.empty((count, n, h), dtype=np.float32)
    edge_used = np.zeros(count, dtype=bool)
    triangle_used = np.zeros(count, dtype=bool)
    valid_edges = np.zeros(count, dtype=np.int64)
    valid_triangles = np.zeros(count, dtype=np.int64)
    for s in range(count):
        r = candidates[s].reshape(m, dim).astype(np.float64)
        b = base_residual[s].reshape(dim).astype(np.float64)
        gram = r @ r.T
        diag = np.diag(gram)
        sq = np.maximum(diag[:, None] + diag[None] - 2 * gram, 0.0)
        np.fill_diagonal(sq, np.inf)
        knn = np.argsort(sq, axis=1)[:, :KNN]
        rho = GAMMA * np.median(np.sqrt(np.min(sq, axis=1)))
        graph = np.zeros((m, m), dtype=bool)
        for i in range(m):
            for j in range(i + 1, m):
                if j in knn[i] and i in knn[j] and (math.sqrt(float(sq[i, j])) <= rho):
                    graph[i, j] = graph[j, i] = True
        singleton_d2 = np.sum((r - b[None]) ** 2, axis=1)
        best_idx = int(np.argmin(singleton_d2))
        p_edge = r[best_idx].copy()
        d2_edge = float(singleton_d2[best_idx])
        used_edge = False
        for i, j in zip(*np.where(np.triu(graph, 1))):
            valid_edges[s] += 1
            a, c = (r[i], r[j])
            v = c - a
            vv = float(v @ v)
            if vv <= 1e-12:
                continue
            t = float(np.clip((b - a) @ v / vv, 0.0, 1.0))
            p = a + t * v
            d2 = float(np.sum((b - p) ** 2))
            if d2 < d2_edge:
                p_edge, d2_edge, used_edge = (p, d2, True)
        p_complex = p_edge.copy()
        d2_complex = d2_edge
        used_triangle = False
        for i, j, k in itertools.combinations(range(m), 3):
            if not (graph[i, j] and graph[i, k] and graph[j, k]):
                continue
            valid_triangles[s] += 1
            a = r[i]
            v1 = r[j] - a
            v2 = r[k] - a
            g11 = float(v1 @ v1)
            g12 = float(v1 @ v2)
            g22 = float(v2 @ v2)
            det = g11 * g22 - g12 * g12
            if det <= 1e-10 * max(g11 * g22, 1.0):
                continue
            rhs1 = float((b - a) @ v1)
            rhs2 = float((b - a) @ v2)
            t1 = (rhs1 * g22 - rhs2 * g12) / det
            t2 = (rhs2 * g11 - rhs1 * g12) / det
            w0 = 1.0 - t1 - t2
            if min(w0, t1, t2) <= 1e-08:
                continue
            p = a + t1 * v1 + t2 * v2
            d2 = float(np.sum((b - p) ** 2))
            if d2 < d2_complex:
                p_complex, d2_complex, used_triangle = (p, d2, True)
        edge_output[s] = p_edge.reshape(n, h).astype(np.float32)
        complex_output[s] = p_complex.reshape(n, h).astype(np.float32)
        edge_used[s] = used_edge
        triangle_used[s] = used_triangle
    return {'edge': edge_output, 'complex': complex_output, 'edge_used': edge_used, 'triangle_used': triangle_used, 'valid_edges': valid_edges, 'valid_triangles': valid_triangles}

def metrics(pred, truth):
    error = pred.astype(np.float64) - truth.astype(np.float64)
    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(np.mean(error * error)))
    wmape = float(np.sum(np.abs(error)) / np.sum(np.abs(truth.astype(np.float64))) * 100.0)
    return (mae, rmse, wmape)

def print_metrics(label, pred, truth):
    mae, rmse, wmape = metrics(pred, truth)
    return (mae, rmse, wmape)


In [ ]:
ft = make_time_features(train_data['future_time']).reshape(len(x_tra), OUT_STEPS, -1)
features = retrieval_features(x_tra, ft)
feature_mean = features.mean(0, keepdims=True)
feature_std = np.maximum(features.std(0, keepdims=True), 1e-06)
np.savez_compressed(BANK_PATH, memory_residual=(y_tra - x_tra[:, :, -1:]).astype(np.float32), memory_retrieval_feat=((features - feature_mean) / feature_std).astype(np.float32), retrieval_feat_mean=feature_mean, retrieval_feat_std=feature_std, train_only=np.array([True]))


In [ ]:
import hashlib

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for z in iter(lambda: f.read(1024 * 1024), b''):
            h.update(z)
    return h.hexdigest()
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()
checkpoint_hash = file_hash(BEST_MODEL_PATH)
bank_hash = file_hash(BANK_PATH)
bank = np.load(BANK_PATH, allow_pickle=False)

@torch.no_grad()
def infer(loader):
    model.eval()
    return np.concatenate([model(batch[0].to(DEVICE), batch[1].to(DEVICE)).cpu().numpy() for batch in loader])

def project_all(base, x, times, include_original=False):
    ft = make_time_features(times).reshape(len(x), OUT_STEPS, -1)
    candidates = retrieve(bank, x, ft)
    last = x[:, :, -1:]
    residual = base - last
    B, K, N, H = candidates.shape
    scale = np.maximum(bank['memory_residual'].std(axis=(0, 2)), 1.0)
    scale = np.maximum(scale, np.median(scale) * 0.1) ** 0.5
    global_weighted = last + project_edges_and_triangles(residual / scale[None, :, None], candidates / scale[None, None, :, None])['complex'] * scale[None, :, None]
    node = last + project_edges_and_triangles(residual.reshape(B * N, 1, H), candidates.transpose(0, 2, 1, 3).reshape(B * N, K, 1, H))['complex'].reshape(B, N, H)
    original = last + project_edges_and_triangles(residual, candidates)['complex'] if include_original else None
    return (global_weighted, node, original)

def statistical_fusion(base, global_projection, node_projection):
    base, pg, pn = (np.asarray(v, dtype=np.float32) for v in (base, global_projection, node_projection))
    dg = np.mean(np.abs(pg - base), axis=-1, keepdims=True)
    dn = np.mean(np.abs(pn - base), axis=-1, keepdims=True)
    den = dg + dn
    w = np.full_like(den, 0.5)
    np.divide(dn, den, out=w, where=den > 1e-20)
    return ((w * pg + (1 - w) * pn).astype(np.float32), w)

def evaluate(pred, truth):
    result = {}
    for label, steps in [('first4', slice(0, 4)), ('full8', slice(None))]:
        t = truth[:, :, steps].astype(np.float64)
        e = pred[:, :, steps].astype(np.float64) - t
        result[label] = dict(MAE=float(np.abs(e).mean()), RMSE=float(np.sqrt((e * e).mean())), WMAPE_percent=float(np.where(t != 0, np.abs(e), 0).sum() / t.sum() * 100))
    return result


In [ ]:
frozen = dict(checkpoint=str(BEST_MODEL_PATH), checkpoint_sha256=checkpoint_hash, bank_sha256=bank_hash, calibration='none: training-free statistical fusion; no validation-label fitting', method='weighted global + node trajectory projection, small_correction_l1', weight_rule='dg=mean_h(abs(pg-base)); dn=mean_h(abs(pn-base)); wg=dn/(dg+dn); wn=1-wg', zero_displacement_rule='wg=wn=0.5 when dg+dn <= 1e-20', weight_shape='[samples,nodes,1], shared across prediction horizons', weight_inputs=['base', 'global_projection', 'node_projection'], direct_base_coefficient=0, wmape='original scenic convention: zero-target errors excluded from numerator')
(EXPERIMENT_DIR / 'frozen_before_test.json').write_text(json.dumps(frozen, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
test_loader = make_loader(test_data, with_targets=False)
y_pred_base_8 = infer(test_loader)
y_tst_base_8 = y_tst
x_tst_base_8 = x_tst
time_tst_base_8 = time_tst


In [ ]:
test_pg, test_pn, old_sp = project_all(y_pred_base_8, x_tst_base_8, time_tst_base_8, include_original=True)
new_sp, global_weights = statistical_fusion(y_pred_base_8, test_pg, test_pn)
fusion_weights = np.stack([global_weights, 1 - global_weights], axis=-1)
results = {'Original_Base': evaluate(y_pred_base_8, y_tst_base_8), 'Original_SP': evaluate(old_sp, y_tst_base_8), 'Original_Base_New_RAG': evaluate(new_sp, y_tst_base_8), 'Weighted_global_hard': evaluate(test_pg, y_tst_base_8), 'Node_hard': evaluate(test_pn, y_tst_base_8)}
for name, pred in [('base', y_pred_base_8), ('old_sp', old_sp), ('new_sp', new_sp), ('truth', y_tst_base_8)]:
    np.save(EXPERIMENT_DIR / (name + '.npy'), pred)
(EXPERIMENT_DIR / 'metrics.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
np.savez_compressed(EXPERIMENT_DIR / 'statistical_fusion_predictions.npz', base=y_pred_base_8, global_projection=test_pg, node_projection=test_pn, prediction=new_sp, truth=y_tst_base_8, global_weight=global_weights, fusion_weights=fusion_weights)
weight_summary = dict(global_mean=float(global_weights.mean()), global_quantiles=np.quantile(global_weights, [0, 0.05, 0.5, 0.95, 1]).tolist(), shape=list(fusion_weights.shape), order=['global', 'node'], note='Summary only; inference uses individual sample/node weights, not this mean.')
(EXPERIMENT_DIR / 'statistical_weight_summary.json').write_text(json.dumps(weight_summary, ensure_ascii=False, indent=2), encoding='utf-8')
